In [ ]:
# ============================================================
# EVALUATION PIPELINE
# ============================================================

import os, json
import pandas as pd
import numpy as np

# -----------------------------
# Agent runs to compare (rows)
# -----------------------------
AGENT_JSON_CANDIDATES = {
    "testt":   ["./testt.json",   "/mnt/data/qwen_formal11.json"],

}

# -----------------------------
# Oracle (optional)
# -----------------------------
ORACLE_JSON_CANDS = ["./test.json", "/mnt/data/44fix.json"]

# -----------------------------
# Output CSV
# -----------------------------
OUT_CSV = "./train12.csv"  # set None to disable writing

def _pick_existing(cands):
    return next((p for p in cands if p and os.path.exists(p)), None)

oracle_path = _pick_existing(ORACLE_JSON_CANDS)

oracle_runs = None
oracle_by_id = None
oracle_by_base = None
if oracle_path is not None:
    with open(oracle_path, "r", encoding="utf-8") as f:
        oracle_runs = json.load(f)

# -----------------------------
# Helpers
# -----------------------------
def ckey(c):
    """Constraint key for comparison."""
    if not isinstance(c, dict) or not c:
        return None
    return (c.get("column"), c.get("op"), str(c.get("value")))

def car_key(car):
    """Car key for comparison (top-1 match)."""
    if not isinstance(car, dict) or not car:
        return None
    return (car.get("Make"), car.get("Model"), str(car.get("Year")), str(car.get("MSRP")))

def normalize_relaxed_constraint(rc):
    """
    Normalize relaxed_constraint to a plain {column, op, value} dict (or None).
    Supports:
      - Plain constraint dict: {"column","op","value"}
      - Repair dict (formal): {"action","from", "to"} where from/to are constraints or None
        Rule: use 'to' if it's a dict (replace), else use 'from' (drop).
    """
    if rc is None:
        return None
    if not isinstance(rc, dict) or not rc:
        return None

    # Already a plain constraint
    if {"column", "op", "value"}.issubset(rc.keys()):
        return {"column": rc.get("column"), "op": rc.get("op"), "value": rc.get("value")}

    # Repair object style (formal)
    if "action" in rc and ("from" in rc or "to" in rc):
        to_c = rc.get("to")
        fr_c = rc.get("from")
        chosen = to_c if isinstance(to_c, dict) and {"column","op","value"}.issubset(to_c.keys()) else fr_c
        if isinstance(chosen, dict) and {"column","op","value"}.issubset(chosen.keys()):
            return {"column": chosen.get("column"), "op": chosen.get("op"), "value": chosen.get("value")}
        return None

    # Unknown dict shape
    return None

def get_oracle_relaxed(b):
    """Support multiple oracle fields and normalize them."""
    if not isinstance(b, dict):
        return None
    raw = (b.get("unique_repair_constraint")
           or b.get("chosen_relaxation")
           or b.get("relaxed_constraint"))
    return normalize_relaxed_constraint(raw)

def build_oracle_index(oracle_list):
    by_id, by_base = {}, {}
    for r in oracle_list or []:
        if not isinstance(r, dict):
            continue
        if "id" in r:
            try:
                by_id[int(r["id"])] = r
            except Exception:
                pass
        if "base_query_sentence" in r:
            by_base[str(r["base_query_sentence"])] = r
    return by_id, by_base

if oracle_runs is not None:
    oracle_by_id, oracle_by_base = build_oracle_index(oracle_runs)

def lookup_oracle(agent_rec):
    if oracle_runs is None:
        return None
    if "id" in agent_rec and oracle_by_id is not None:
        try:
            aid = int(agent_rec["id"])
            if aid in oracle_by_id:
                return oracle_by_id[aid]
        except Exception:
            pass
    bq = agent_rec.get("base_query_sentence")
    if bq is not None and oracle_by_base is not None and str(bq) in oracle_by_base:
        return oracle_by_base[str(bq)]
    return None

def safe_rate(num, den):
    return (num / den) if den else np.nan

def fmt_pct(x, decimals=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{x*100:.{decimals}f}%"

def load_agent_runs(path):
    """
    Robust loader:
      - If JSON is a list: treat as runs
      - If JSON is a dict: try common container keys
    """
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if isinstance(obj, list):
        return obj

    if isinstance(obj, dict):
        for k in ["agent_runs", "runs", "records", "items", "data", "outputs", "results"]:
            v = obj.get(k)
            if isinstance(v, list):
                return v

    raise ValueError(f"Unrecognized agent JSON schema in {path}: expected list or dict containing a list of runs.")

# -----------------------------
# Core evaluation per file
# -----------------------------
def eval_one(agent_path):
    agent_runs = load_agent_runs(agent_path)
    N = len(agent_runs)

    status_counts = pd.Series([r.get("status") for r in agent_runs]).value_counts(dropna=False)

    # Dialogue completion (overall)
    total_slots = 0
    total_parsed = 0
    slots_lens, parsed_lens = [], []

    for r in agent_runs:
        slots = r.get("slots_provided_to_agent") or []
        parsed = r.get("parsed_additional_constraints") or []
        total_slots += len(slots)
        total_parsed += len(parsed)
        slots_lens.append(len(slots))
        parsed_lens.append(len(parsed))

    avg_slots = float(np.mean(slots_lens)) if N else 0.0
    avg_parsed = float(np.mean(parsed_lens)) if N else 0.0
    slot_completion_overall = safe_rate(total_parsed, total_slots)

    # Recommendation produced
    rec_present = sum(
        1 for r in agent_runs
        if isinstance(r.get("recommended_car"), dict) and r.get("recommended_car")
    )
    rec_rate = safe_rate(rec_present, N)

    # Oracle agreement (optional)
    oracle_comparable = 0
    relax_match = 0
    car_match_relax_gated = 0  # <-- NEW: relax AND car match, denominator oracle_comparable

    if oracle_runs is not None:
        for r in agent_runs:
            b = lookup_oracle(r)
            if b is None:
                continue

            agent_relaxed_norm = normalize_relaxed_constraint(r.get("relaxed_constraint"))
            agent_car = r.get("recommended_car")

            oracle_relaxed_norm = get_oracle_relaxed(b)
            oracle_car = b.get("recommended_car")

            if oracle_relaxed_norm is None:
                continue

            oracle_comparable += 1

            is_relax_ok = (ckey(agent_relaxed_norm) == ckey(oracle_relaxed_norm))
            relax_match += int(is_relax_ok)

            is_car_ok = (car_key(agent_car) == car_key(oracle_car))
            car_match_relax_gated += int(is_relax_ok and is_car_ok)

    out = {
        "N": N,
        "Avg #Slots Provided": avg_slots,
        "Avg #Constraints Parsed": avg_parsed,
        "Slot Completion Rate (overall)": slot_completion_overall,
        "SAT (No Relaxation) Rate": safe_rate(int(status_counts.get("SAT_no_relaxation", 0)), N),
        "SAT (After Relaxation) Rate": safe_rate(int(status_counts.get("SAT_after_relaxation", 0)), N),
        "UNSAT Rate": safe_rate(int(status_counts.get("UNSAT_even_after_relaxation", 0)), N),
        "Recommendation Produced Rate": rec_rate,
    }

    if oracle_runs is not None:
        out.update({
            "Comparable (oracle has relaxed constraint)": oracle_comparable,
            "Relax Match Rate": safe_rate(relax_match, oracle_comparable),
            "Recommended Car Match Rate (Relax-Gated)": safe_rate(car_match_relax_gated, oracle_comparable),
        })

    return out

# -----------------------------
# Run + build summary table
# -----------------------------
rows = []
resolved_paths = {}

for approach, cands in AGENT_JSON_CANDIDATES.items():
    p = _pick_existing(cands)
    if p is None:
        raise FileNotFoundError(f"Could not find {approach}. Tried: {cands}")
    resolved_paths[approach] = p

    metrics = eval_one(p)
    metrics["Approach"] = approach
    rows.append(metrics)

df = pd.DataFrame(rows).set_index("Approach")

# Format: percentages for rate columns, keep counts/averages numeric
rate_cols = [
    "Slot Completion Rate (overall)",
    "SAT (No Relaxation) Rate",
    "SAT (After Relaxation) Rate",
    "UNSAT Rate",
    "Recommendation Produced Rate",
    "Relax Match Rate",
    "Recommended Car Match Rate (Relax-Gated)",
]
for c in rate_cols:
    if c in df.columns:
        df[c] = df[c].apply(lambda x: fmt_pct(x, decimals=1))

# Optional rounding for averages
for c in ["Avg #Slots Provided", "Avg #Constraints Parsed"]:
    if c in df.columns:
        df[c] = df[c].astype(float).round(3)

# Ensure N / Comparable are ints where present
for c in ["N", "Comparable (oracle has relaxed constraint)"]:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype(int)

# -----------------------------
# Print + Save CSV
# -----------------------------
print("=== EVAL INPUTS ===")
print("Agent files used:")
for k, v in resolved_paths.items():
    print(f" - {k}: {v}")
print("Oracle benchmark:", oracle_path if oracle_path is not None else "(not found; oracle metrics skipped)")
print()
print("=== SUMMARY METRICS TABLE (rows=approaches, cols=metrics) ===")
print(df.to_string())

if OUT_CSV:
    df.reset_index().to_csv(OUT_CSV, index=False)
    print("\nWrote summary CSV:", OUT_CSV)
